#  Topic Modeling and Trend Detection in Large Text Corpora

Topic modeling is a technique in natural language processing (NLP) and machine learning that aims to uncover latent thematic structures within a collection of texts. Topic modelling is a system learning technique that robotically discovers the principle themes or "topics" representing a huge document collection. The intention of topic modelling is to discover the hidden semantic systems within textual content facts, permitting customers to arrange, apprehend, and summarize the data in a manner that is each green and insightful.




## Arxiv Dataset
For this project we will use the Arxiv dataset which is a mirror of the original ArXiv data. For nearly 30 years, ArXiv has served the public and research communities by providing open access to scholarly articles, from the vast branches of physics to the many subdisciplines of computer science to everything in between, including math, statistics, electrical engineering, quantitative biology, and economics. This rich corpus of information offers significant, but sometimes overwhelming depth. Since the full arXiv dataset is quite large (approximately 1.1 TB and continuously growing), for the purposes of this project I will be using only the first 40,000 rows to reduce resource usage and ensure faster processing during development and experimentation.

### Install Bertopic

In [ ]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-c

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Import the necessary libraries

In [ ]:
from umap import UMAP
import pandas as pd
from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
import numpy as np
from bertopic import BERTopic
from sklearn.preprocessing import OneHotEncoder
from bertopic.representation import KeyBERTInspired,MaximalMarginalRelevance
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.metrics import silhouette_score,calinski_harabasz_score,classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

### Fetch dataset

In [ ]:
dataset = pd.read_csv('/content/drive/MyDrive/arxiv-40k.csv')

In [ ]:
dataset

,update_date,title,journal-ref,submitter,authors,doi,comments,categories,abstract,id,license,report-no,versions,authors_parsed
0,2008-11-26,Calculation of prompt diphoton production cros...,"Phys.Rev.D76:013009,2007",Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",10.1103/PhysRevD.76.013009,"37 pages, 15 figures; published version",hep-ph,A fully differential calculation in perturba...,704.0001,NaN,ANL-HEP-PR-07-12,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...","[['Balázs', 'C.', ''], ['Berger', 'E. L.', '']..."
1,2008-12-13,Sparsity-certifying Graph Decompositions,NaN,Louis Theran,Ileana Streinu and Louis Theran,NaN,To appear in Graphs and Combinatorics,math.CO cs.CG,"We describe a new algorithm, the $(k,\ell)$-...",704.0002,http://arxiv.org/licenses/nonexclusive-distrib...,NaN,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...","[['Streinu', 'Ileana', ''], ['Theran', 'Louis'..."
2,2008-01-13,The evolution of the Earth-Moon system based o...,NaN,Hongjun Pan,Hongjun Pan,NaN,"23 pages, 3 figures",physics.gen-ph,The evolution of Earth-Moon system is descri...,704.0003,NaN,NaN,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...","[['Pan', 'Hongjun', '']]"
3,2007-05-23,A determinant of Stirling cycle numbers counts...,NaN,David Callan,David Callan,NaN,11 pages,math.CO,We show that a determinant of Stirling cycle...,704.0004,NaN,NaN,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...","[['Callan', 'David', '']]"
4,2013-10-15,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,"Illinois J. Math. 52 (2008) no.2, 681-689",Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,NaN,NaN,math.CA math.FA,In this paper we show how to compute the $\L...,704.0005,NaN,NaN,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...","[['Abu-Shammala', 'Wael', ''], ['Torchinsky', ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,2008-11-26,Flavorful Supersymmetry,"Phys.Rev.D77:075006,2008",Yasunori Nomura,"Yasunori Nomura, Michele Papucci, Daniel Stola...",10.1103/PhysRevD.77.075006,"20 pages; typos corrected, comments added, to ...",hep-ph,Weak scale supersymmetry provides elegant so...,712.2074,NaN,UCB-PTH-07/25,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Nomura', 'Yasunori', ''], ['Papucci', 'Mich..."
39996,2011-07-20,"($\ell,0)$-Carter partitions, a generating fun...","Electronic Journal of Combinatorics, Volume 15...",Chris Berg,"Chris Berg, Monica Vazirani",NaN,NaN,math.CO math.RT,In this paper we give an alternate combinato...,712.2075,http://arxiv.org/licenses/nonexclusive-distrib...,NaN,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Berg', 'Chris', ''], ['Vazirani', 'Monica',..."
39997,2007-12-20,On the irreducible representations of a finite...,NaN,Benjamin Steinberg,"Olexandr Ganyushkin, Volodymyr Mazorchuk and B...",NaN,NaN,math.RT math.GR,"Work of Clifford, Munn and Ponizovski{\u\i} ...",712.2076,NaN,NaN,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Ganyushkin', 'Olexandr', ''], ['Mazorchuk',..."
39998,2009-11-13,The Formation of Constellation III in the Larg...,NaN,Jason Harris,Jason Harris and Dennis Zaritsky,10.1071/AS07037,Accepted for publication in the Publications o...,astro-ph,We present a detailed reconstruction of the ...,712.2077,NaN,NaN,"[{'version': 'v1', 'created': 'Thu, 13 Dec 200...","[['Harris', 'Jason', ''], ['Zaritsky', 'Dennis..."


### Data Preprocessing

In [ ]:
dataset.isnull().sum()

,0
update_date,0
title,0
journal-ref,19270
submitter,0
authors,0
doi,15629
comments,4709
categories,0
abstract,0
id,0


In [ ]:
dataset = dataset.drop(columns=['update_date','title','journal-ref','submitter','authors','doi','comments','id','license','report-no','versions','authors_parsed'],axis=1)
dataset

,categories,abstract
0,hep-ph,A fully differential calculation in perturba...
1,math.CO cs.CG,"We describe a new algorithm, the $(k,\ell)$-..."
2,physics.gen-ph,The evolution of Earth-Moon system is descri...
3,math.CO,We show that a determinant of Stirling cycle...
4,math.CA math.FA,In this paper we show how to compute the $\L...
...,...,...
39995,hep-ph,Weak scale supersymmetry provides elegant so...
39996,math.CO math.RT,In this paper we give an alternate combinato...
39997,math.RT math.GR,"Work of Clifford, Munn and Ponizovski{\u\i} ..."
39998,astro-ph,We present a detailed reconstruction of the ...


In [ ]:
threshold = 334
val_counts = dataset['categories'].value_counts()
rare_categories = val_counts[val_counts < threshold].index
dataset['categories'] = dataset['categories'].apply(
    lambda x: 'other' if x in rare_categories else x
)

In [ ]:
dataset['categories'].value_counts()

,count
categories,
other,20024
astro-ph,6720
hep-ph,2222
quant-ph,1723
hep-th,1525
cond-mat.mtrl-sci,843
gr-qc,804
cond-mat.mes-hall,697
hep-ex,632


In [ ]:
len(dataset['categories'].value_counts())

20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    dataset['abstract'], dataset['categories'], test_size=0.2,
    stratify=dataset['categories'], random_state=42
)

## Create the BertTopic unsupervised Model 1

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15,metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words="english")

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic representations
)

In [ ]:
topics,probs = topic_model.fit_transform(X_train.tolist())

### Evaluation on the BertTopic model

In [ ]:
embeddings = topic_model.umap_model.embedding_
mask = np.array(topics) != -1
filtered_topics = np.array(topics)[mask]

print(f"Silhouette: {silhouette_score(embeddings[mask], filtered_topics):.3f}")
print(f"CH Index: {calinski_harabasz_score(embeddings[mask], filtered_topics):.0f}")

Silhouette: 0.536
CH Index: 22889


### The most frequent topics

In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,14677,-1_gamma_transition_evolution_dynamics,"[gamma, transition, evolution, dynamics, densi...",[ Motivated by the possibility of creating no...
1,0,1257,0_quantum_bosons_entanglement_scattering,"[quantum, bosons, entanglement, scattering, ha...",[ The physics of quantum degenerate Fermi gas...
2,1,413,1_minimax_asymptotic_models_likelihood,"[minimax, asymptotic, models, likelihood, risk...",[ We consider the problem of estimating the u...
3,2,353,2_scattering_microscopic_particle_coulomb,"[scattering, microscopic, particle, coulomb, d...",[ Properties of effective interactions in neu...
4,3,315,3_galactic_spectral_spectra_flux,"[galactic, spectral, spectra, flux, luminosity...",[ We report on two Chandra observations of th...
...,...,...,...,...,...
228,227,15,227_dynamical_bound_models_gravitationally,"[dynamical, bound, models, gravitationally, co...","[ In young star clusters, the density can be ..."
229,228,15,228_boron_flux_superconductivity_reactive,"[boron, flux, superconductivity, reactive, tc,...",[ The effect of the quality of starting powde...
230,229,15,229_worlds_renormalizable_evolutions_doublets,"[worlds, renormalizable, evolutions, doublets,...",[ We present a simple and realistic model of ...
231,230,15,230_cosmological_cosmic_stars_intergalactic,"[cosmological, cosmic, stars, intergalactic, s...",[ We present numerical simulations of how a 1...


### Frequent words from the second most frequent topic

In [ ]:
topic_model.get_topic(0)

[('quantum', np.float32(0.24134642)),
 ('bosons', np.float32(0.2407646)),
 ('entanglement', np.float32(0.2400794)),
 ('scattering', np.float32(0.21828768)),
 ('harmonic', np.float32(0.19498764)),
 ('potential', np.float32(0.19493227)),
 ('trapping', np.float32(0.18285118)),
 ('fermions', np.float32(0.18015389)),
 ('lattices', np.float32(0.16824923)),
 ('excitation', np.float32(0.16710997))]

### Info about the documents clustered in these topics

In [ ]:
topic_model.get_document_info(X_train)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,A detailed comparison of the expressions for...,2,2_scattering_microscopic_particle_coulomb,"[scattering, microscopic, particle, coulomb, d...",[ Properties of effective interactions in neu...,scattering - microscopic - particle - coulomb ...,1.000000,False
1,This is an expanded version of our earlier p...,58,58_conjectures_asymptotic_conjecture_infinitely,"[conjectures, asymptotic, conjecture, infinite...",[ A classical problem in analytic number theo...,conjectures - asymptotic - conjecture - infini...,0.234291,False
2,Solutions to conservation laws satisfy the m...,-1,-1_gamma_transition_evolution_dynamics,"[gamma, transition, evolution, dynamics, densi...",[ Motivated by the possibility of creating no...,gamma - transition - evolution - dynamics - de...,0.000000,False
3,By means of the 10-resonance unitary and ana...,127,127_renormalization_quark_nucleon_chiral,"[renormalization, quark, nucleon, chiral, hype...",[ We study a new nucleon resonance from eta p...,renormalization - quark - nucleon - chiral - h...,1.000000,False
4,A pedigree is a directed graph that describe...,-1,-1_gamma_transition_evolution_dynamics,"[gamma, transition, evolution, dynamics, densi...",[ Motivated by the possibility of creating no...,gamma - transition - evolution - dynamics - de...,0.000000,False
...,...,...,...,...,...,...,...,...
31995,The high pressure phase diagram of CsC8 grap...,5,5_symmetry_scattering_nanoribbon_fermions,"[symmetry, scattering, nanoribbon, fermions, n...",[ We report the existence of zero energy surf...,symmetry - scattering - nanoribbon - fermions ...,0.338843,False
31996,We present a preliminary set of updated NLO ...,106,106_qcd_photons_hadron_photon,"[qcd, photons, hadron, photon, data, lhc, tran...",[ With the knowledge of diffractive parton de...,qcd - photons - hadron - photon - data - lhc -...,0.634735,False
31997,We have investigated the doping dependence o...,19,19_isotope_spectroscopy_scattering_observed,"[isotope, spectroscopy, scattering, observed, ...",[ Pairing of electrons in conventional superc...,isotope - spectroscopy - scattering - observed...,0.667108,False
31998,We calculate the O($\eps$)--term of the two-...,110,110_feynman_quark_qcd_renormalization,"[feynman, quark, qcd, renormalization, propaga...",[ The dynamically generated effective gluon m...,feynman - quark - qcd - renormalization - prop...,0.530741,False


## Data Visualization of the topics

In [ ]:
topic_model.visualize_barchart(top_n_topics=20,n_words=10)

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topics_per_class = topic_model.topics_per_class(X_train, classes=y_train)
topic_model.visualize_topics_per_class(topics_per_class)

In [ ]:
from collections import Counter
import numpy as np


# 2️⃣ Map each topic to the most frequent class in that topic
topic_to_class = {}
for topic_id in set(topics):
    if topic_id == -1:  # -1 is usually "outlier topic" in BERTopic
        continue
    # Get indices of documents in this topic
    idxs = [i for i, t in enumerate(topics) if t == topic_id]
    # Get their classes
    classes_in_topic = [y_train.iloc[i] for i in idxs]
    # Find the most common class
    most_common_class = Counter(classes_in_topic).most_common(1)[0][0]
    topic_to_class[topic_id] = most_common_class

# 3️⃣ Show mapping
for t, c in topic_to_class.items():
    print(f"Topic {t} → Class '{c}'")

Topic 0 → Class 'other'
Topic 1 → Class 'other'
Topic 2 → Class 'nucl-th'
Topic 3 → Class 'astro-ph'
Topic 4 → Class 'hep-ph'
Topic 5 → Class 'other'
Topic 6 → Class 'math.AG'
Topic 7 → Class 'other'
Topic 8 → Class 'cs.IT math.IT'
Topic 9 → Class 'other'
Topic 10 → Class 'other'
Topic 11 → Class 'other'
Topic 12 → Class 'hep-th'
Topic 13 → Class 'other'
Topic 14 → Class 'cond-mat.mes-hall'
Topic 15 → Class 'astro-ph'
Topic 16 → Class 'cond-mat.mes-hall'
Topic 17 → Class 'other'
Topic 18 → Class 'cond-mat.stat-mech'
Topic 19 → Class 'other'
Topic 20 → Class 'astro-ph'
Topic 21 → Class 'other'
Topic 22 → Class 'astro-ph'
Topic 23 → Class 'other'
Topic 24 → Class 'other'
Topic 25 → Class 'other'
Topic 26 → Class 'other'
Topic 27 → Class 'astro-ph'
Topic 28 → Class 'quant-ph'
Topic 29 → Class 'astro-ph'
Topic 30 → Class 'astro-ph'
Topic 31 → Class 'quant-ph'
Topic 32 → Class 'other'
Topic 33 → Class 'astro-ph'
Topic 34 → Class 'hep-ex'
Topic 35 → Class 'other'
Topic 36 → Class 'other'
Top

In [ ]:
mapped_train_labels = [topic_to_class.get(t, "Unknown") for t in topics]

# Prepare topic IDs as feature for classifier
X_topics_train = pd.DataFrame(topics, columns=["topic"])
encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded = encoder.fit_transform(X_topics_train)


In [ ]:
X_topics_train

,topic
0,2
1,58
2,-1
3,127
4,-1
...,...
31995,5
31996,106
31997,19
31998,110


In [ ]:
clf = RandomForestClassifier(n_estimators=200,criterion='entropy', random_state=42)
clf.fit(X_train_encoded, mapped_train_labels)

RandomForestClassifier(criterion='entropy', n_estimators=200, random_state=42)

In [ ]:
topics_test, _ = topic_model.transform(X_test.tolist())
X_test_topics = pd.DataFrame(topics_test, columns=["topic"])
X_test_encoded = encoder.transform(X_test_topics)

# Predict classes on test set
y_pred = clf.predict(X_test_encoded)

# Evaluate
print(classification_report(y_test, y_pred))

                    precision    recall  f1-score   support

           Unknown       0.00      0.00      0.00         0
          astro-ph       0.92      0.44      0.59      1344
 cond-mat.mes-hall       0.49      0.26      0.34       139
 cond-mat.mtrl-sci       0.45      0.05      0.10       169
    cond-mat.other       0.00      0.00      0.00        90
cond-mat.stat-mech       0.57      0.24      0.33       106
   cond-mat.str-el       0.65      0.30      0.41       115
     cs.IT math.IT       0.70      0.76      0.73        67
             gr-qc       0.43      0.19      0.26       161
            hep-ex       0.80      0.37      0.51       126
           hep-lat       0.60      0.21      0.32        70
            hep-ph       0.78      0.23      0.35       444
            hep-th       0.67      0.23      0.35       305
   math-ph math.MP       0.00      0.00      0.00        93
           math.AG       0.67      0.38      0.48        79
           math.CO       0.42      0.07

In [ ]:
clf = DecisionTreeClassifier(criterion='entropy', random_state=42)
clf.fit(X_train_encoded, mapped_train_labels)

DecisionTreeClassifier(criterion='entropy', random_state=42)

In [ ]:
topics_test, _ = topic_model.transform(X_test.tolist())
X_test_topics = pd.DataFrame(topics_test, columns=["topic"])
X_test_encoded = encoder.transform(X_test_topics)

# Predict classes on test set
y_pred = clf.predict(X_test_encoded)

# Evaluate
print(classification_report(y_test, y_pred))

                    precision    recall  f1-score   support

           Unknown       0.00      0.00      0.00         0
          astro-ph       0.92      0.44      0.59      1344
 cond-mat.mes-hall       0.49      0.26      0.34       139
 cond-mat.mtrl-sci       0.45      0.05      0.10       169
    cond-mat.other       0.00      0.00      0.00        90
cond-mat.stat-mech       0.57      0.24      0.33       106
   cond-mat.str-el       0.65      0.30      0.41       115
     cs.IT math.IT       0.70      0.76      0.73        67
             gr-qc       0.43      0.19      0.26       161
            hep-ex       0.80      0.37      0.51       126
           hep-lat       0.60      0.21      0.32        70
            hep-ph       0.78      0.23      0.35       444
            hep-th       0.67      0.23      0.35       305
   math-ph math.MP       0.00      0.00      0.00        93
           math.AG       0.67      0.38      0.48        79
           math.CO       0.42      0.07

## Create BertTopic Model unsupervised 2

In [ ]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")


# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15,allow_single_cluster=True,metric='euclidean', algorithm='best', cluster_selection_method='eom', prediction_data=True)

# Step 4 - Tokenize topics
vectorizer_model = TfidfVectorizer(stop_words="english")

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()
# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model_2 = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model = ctfidf_model,
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic representations
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
topics, _ = topic_model_2.fit_transform(X_train.tolist())

### Evaluation

In [ ]:
embeddings = topic_model_2.umap_model.embedding_
mask = np.array(topics) != -1
filtered_topics = np.array(topics)[mask]

print(f"Silhouette: {silhouette_score(embeddings[mask], filtered_topics):.3f}")
print(f"CH Index: {calinski_harabasz_score(embeddings[mask], filtered_topics):.0f}")

Silhouette: 0.578
CH Index: 6958


### The most frequent topics

In [ ]:
topic_model_2.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,11184,-1_nucleon_proton_quark_finite,"[nucleon, proton, quark, finite, qcd, lhc, dec...",[ We investigate whether a mass scale for ele...
1,0,5200,0_abundances_observations_redshifts_observed,"[abundances, observations, redshifts, observed...",[ Elliptical galaxies probably host the most ...
2,1,925,1_conformational_conformation_particles_disloc...,"[conformational, conformation, particles, disl...","[ Soft materials, such as colloidal suspensio..."
3,2,724,2_fermions_fermionic_solitons_quantum,"[fermions, fermionic, solitons, quantum, super...",[ We study the effect of a one dimensional op...
4,3,556,3_superconductivity_superconductors_supercondu...,"[superconductivity, superconductors, supercond...",[ We analyze the ground state properties of a...
...,...,...,...,...,...
206,205,16,205_fermions_spinons_quantization_fermionic,"[fermions, spinons, quantization, fermionic, f...",[ The (discrete) Gross-Neveu model is studied...
207,206,16,206_pulsars_hadronic_gravitational_ellipticity,"[pulsars, hadronic, gravitational, ellipticity...","[ Recently, observations of compact stars hav..."
208,207,15,207_quantum_flaws_qubits_3db,"[quantum, flaws, qubits, 3db, entanglement, qc...",[ We analyze a possibility of using the two q...
209,208,15,208_atoms_isomers_pseudopotential_orbitals,"[atoms, isomers, pseudopotential, orbitals, an...","[ Theoretical studies on the geometry, electr..."


### Frequent words from the second most frequent topic

In [ ]:
topic_model_2.get_topic(0)

[('abundances', np.float32(0.21294507)),
 ('observations', np.float32(0.1940336)),
 ('redshifts', np.float32(0.18382359)),
 ('observed', np.float32(0.17563762)),
 ('abundance', np.float32(0.16644563)),
 ('quasars', np.float32(0.16623583)),
 ('variability', np.float32(0.16139035)),
 ('massive', np.float32(0.15517385)),
 ('kpc', np.float32(0.13751893)),
 ('stars', np.float32(0.13693151))]

### Info about the documents clustered in these topics

In [ ]:
topic_model_2.get_document_info(X_train)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,A detailed comparison of the expressions for...,-1,-1_nucleon_proton_quark_finite,"[nucleon, proton, quark, finite, qcd, lhc, dec...",[ We investigate whether a mass scale for ele...,nucleon - proton - quark - finite - qcd - lhc ...,0.000000,False
1,This is an expanded version of our earlier p...,41,41_pseudosquares_q_k_sums_psi_n,"[pseudosquares, q_k, sums, psi_n, sumset, p_2,...",[ We prove that there are infinitely often pa...,pseudosquares - q_k - sums - psi_n - sumset - ...,0.448170,False
2,Solutions to conservation laws satisfy the m...,-1,-1_nucleon_proton_quark_finite,"[nucleon, proton, quark, finite, qcd, lhc, dec...",[ We investigate whether a mass scale for ele...,nucleon - proton - quark - finite - qcd - lhc ...,0.000000,False
3,By means of the 10-resonance unitary and ana...,30,30_parameterizations_polarizabilities_relativi...,"[parameterizations, polarizabilities, relativi...",[ The meson-cloud model of the nucleon consis...,parameterizations - polarizabilities - relativ...,1.000000,False
4,A pedigree is a directed graph that describe...,149,149_recombination_data_reconstructed_vectors,"[recombination, data, reconstructed, vectors, ...",[ Phylogenetic networks are a generalization ...,recombination - data - reconstructed - vectors...,0.378266,False
...,...,...,...,...,...,...,...,...
31995,The high pressure phase diagram of CsC8 grap...,7,7_magnetic_coulomb_dirac_fermi,"[magnetic, coulomb, dirac, fermi, electron, el...",[ We study the effects of metallic doping on ...,magnetic - coulomb - dirac - fermi - electron ...,0.429950,False
31996,We present a preliminary set of updated NLO ...,-1,-1_nucleon_proton_quark_finite,"[nucleon, proton, quark, finite, qcd, lhc, dec...",[ We investigate whether a mass scale for ele...,nucleon - proton - quark - finite - qcd - lhc ...,0.000000,False
31997,We have investigated the doping dependence o...,3,3_superconductivity_superconductors_supercondu...,"[superconductivity, superconductors, supercond...",[ We analyze the ground state properties of a...,superconductivity - superconductors - supercon...,1.000000,False
31998,We calculate the O($\eps$)--term of the two-...,136,136_transcendentality_unitarity_feynman_photon,"[transcendentality, unitarity, feynman, photon...",[ Feynman diagrams may be evaluated by Mellin...,transcendentality - unitarity - feynman - phot...,0.557707,False


## Visualization of the topics

In [ ]:
topic_model_2.visualize_barchart(top_n_topics=20,n_words=8)

In [ ]:
topic_model_2.visualize_heatmap()

In [ ]:
topic_model_2.visualize_topics()

In [ ]:
topic_model_2.visualize_hierarchy()

In [ ]:
topics_per_class = topic_model_2.topics_per_class(X_train, classes=y_train)
topic_model_2.visualize_topics_per_class(topics_per_class)

In [ ]:
topic_to_class = {}
for topic_id in set(topics):
    if topic_id == -1:  # -1 is usually "outlier topic" in BERTopic
        continue
    # Get indices of documents in this topic
    idxs = [i for i, t in enumerate(topics) if t == topic_id]
    # Get their classes
    classes_in_topic = [y_train.iloc[i] for i in idxs]
    # Find the most common class
    most_common_class = Counter(classes_in_topic).most_common(1)[0][0]
    topic_to_class[topic_id] = most_common_class

# 3️⃣ Show mapping
for t, c in topic_to_class.items():
    print(f"Topic {t} → Class '{c}'")

Topic 0 → Class 'astro-ph'
Topic 1 → Class 'other'
Topic 2 → Class 'other'
Topic 3 → Class 'other'
Topic 4 → Class 'other'
Topic 5 → Class 'other'
Topic 6 → Class 'other'
Topic 7 → Class 'other'
Topic 8 → Class 'math.AG'
Topic 9 → Class 'other'
Topic 10 → Class 'other'
Topic 11 → Class 'other'
Topic 12 → Class 'math.PR'
Topic 13 → Class 'other'
Topic 14 → Class 'hep-th'
Topic 15 → Class 'other'
Topic 16 → Class 'quant-ph'
Topic 17 → Class 'hep-lat'
Topic 18 → Class 'other'
Topic 19 → Class 'nucl-th'
Topic 20 → Class 'hep-ph'
Topic 21 → Class 'quant-ph'
Topic 22 → Class 'cond-mat.mes-hall'
Topic 23 → Class 'cond-mat.str-el'
Topic 24 → Class 'hep-ph'
Topic 25 → Class 'other'
Topic 26 → Class 'other'
Topic 27 → Class 'hep-ex'
Topic 28 → Class 'other'
Topic 29 → Class 'other'
Topic 30 → Class 'other'
Topic 31 → Class 'other'
Topic 32 → Class 'cond-mat.stat-mech'
Topic 33 → Class 'other'
Topic 34 → Class 'other'
Topic 35 → Class 'hep-th'
Topic 36 → Class 'other'
Topic 37 → Class 'other'
Top

In [ ]:
mapped_train_labels = [topic_to_class.get(t, "Unknown") for t in topics]

# Prepare topic IDs as feature for classifier
X_topics_train = pd.DataFrame(topics, columns=["topic"])
encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded = encoder.fit_transform(X_topics_train)


In [ ]:
clf = RandomForestClassifier(n_estimators=200,criterion='entropy', random_state=42)
clf.fit(X_train_encoded, mapped_train_labels)

RandomForestClassifier(criterion='entropy', n_estimators=200, random_state=42)

In [ ]:
topics_test, _ = topic_model_2.transform(X_test.tolist())
X_test_topics = pd.DataFrame(topics_test, columns=["topic"])
X_test_encoded = encoder.transform(X_test_topics)

# Predict classes on test set
y_pred = clf.predict(X_test_encoded)

# Evaluate
print(classification_report(y_test, y_pred))

                    precision    recall  f1-score   support

           Unknown       0.00      0.00      0.00         0
          astro-ph       0.94      0.93      0.93      1344
 cond-mat.mes-hall       0.44      0.16      0.23       139
 cond-mat.mtrl-sci       0.67      0.04      0.07       169
    cond-mat.other       0.00      0.00      0.00        90
cond-mat.stat-mech       0.56      0.21      0.30       106
   cond-mat.str-el       0.56      0.25      0.35       115
     cs.IT math.IT       0.69      0.54      0.61        67
             gr-qc       0.57      0.25      0.35       161
            hep-ex       0.63      0.35      0.45       126
           hep-lat       0.64      0.60      0.62        70
            hep-ph       0.78      0.25      0.37       444
            hep-th       0.66      0.24      0.35       305
   math-ph math.MP       0.17      0.01      0.02        93
           math.AG       0.66      0.42      0.51        79
           math.CO       0.56      0.07

In [ ]:
clf = DecisionTreeClassifier(criterion='entropy', random_state=42)
clf.fit(X_train_encoded, mapped_train_labels)

DecisionTreeClassifier(criterion='entropy', random_state=42)

In [ ]:
topics_test, _ = topic_model.transform(X_test.tolist())
X_test_topics = pd.DataFrame(topics_test, columns=["topic"])
X_test_encoded = encoder.transform(X_test_topics)

# Predict classes on test set
y_pred = clf.predict(X_test_encoded)

# Evaluate
print(classification_report(y_test, y_pred))

                    precision    recall  f1-score   support

           Unknown       0.00      0.00      0.00         0
          astro-ph       0.00      0.00      0.00      1344
 cond-mat.mes-hall       0.00      0.00      0.00       139
 cond-mat.mtrl-sci       0.00      0.00      0.00       169
    cond-mat.other       0.00      0.00      0.00        90
cond-mat.stat-mech       0.00      0.00      0.00       106
   cond-mat.str-el       0.00      0.00      0.00       115
     cs.IT math.IT       0.00      0.00      0.00        67
             gr-qc       0.06      0.02      0.03       161
            hep-ex       0.00      0.00      0.00       126
           hep-lat       0.00      0.00      0.00        70
            hep-ph       0.00      0.00      0.00       444
            hep-th       0.03      0.01      0.02       305
   math-ph math.MP       0.00      0.00      0.00        93
           math.AG       0.00      0.00      0.00        79
           math.CO       0.00      0.00